# Faultline: the same Qwen3 model pair on AMD

Run cells in order. This preserves ROCm PyTorch and uses no sudo. It replaces only the model server in `/workspace/faultline-amd` on port 8180. Models download from Hugging Face at the same revisions as both Sparks.


In [ ]:
import json, os, secrets, subprocess, sys, time
from pathlib import Path
setup_dir = Path('/workspace/faultline-amd')
setup_dir.mkdir(exist_ok=True)
(setup_dir / 'amd_model_server.py').write_text('import asyncio\nimport json\nimport os\nimport secrets\nimport time\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\n\nfrom fastapi import Depends, FastAPI, Header, HTTPException\nfrom pydantic import BaseModel, Field\n\nMODEL = os.environ.get("AMD_MODEL", "Qwen/Qwen3-4B")\nDUAL_MODELS = os.environ.get("AMD_DUAL_MODELS", "false").lower() == "true"\nMODEL_MANIFEST = Path(\n    os.environ.get(\n        "AMD_MODEL_MANIFEST", str(Path(__file__).with_name("qwen-models.json"))\n    )\n)\nKEY_FILE = Path(os.environ.get("AMD_KEY_FILE", "/workspace/faultline-amd/api-key"))\nstate = {"ready": False, "active": False, "active_models": set()}\n\n\ndef model_ids():\n    return list(state.get("models", {})) or [MODEL]\n\n\ndef load_model():\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    if not torch.cuda.is_available() or not torch.version.hip:\n        raise RuntimeError("An AMD ROCm GPU is required")\n    catalog = (\n        json.loads(MODEL_MANIFEST.read_text())\n        if DUAL_MODELS\n        else {"all": {"model": MODEL}}\n    )\n    if DUAL_MODELS and torch.cuda.mem_get_info()[0] < 28 * 1024**3:\n        raise RuntimeError(\n            "The two BF16 models require at least 28 GiB free GPU memory before loading"\n        )\n    loaded = {}\n    for group, entry in catalog.items():\n        model_id = entry["model"]\n        revision = entry.get("revision")\n        tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)\n        model = AutoModelForCausalLM.from_pretrained(\n            model_id,\n            revision=revision,\n            torch_dtype=torch.bfloat16,\n            device_map="cuda:0",\n            attn_implementation="sdpa",\n        ).eval()\n        loaded[model_id] = {\n            "tokenizer": tokenizer,\n            "model": model,\n            "revision": revision,\n            "decision_group": group,\n        }\n    state["models"] = loaded\n    state["gpu"] = torch.cuda.get_device_name(0)\n    state["rocm"] = torch.version.hip\n    state["ready"] = True\n\n\n@asynccontextmanager\nasync def lifespan(application):\n    state["key"] = KEY_FILE.read_text().strip()\n    if len(state["key"]) < 32:\n        raise RuntimeError("A strong API key is required")\n    await asyncio.to_thread(load_model)\n    yield\n\n\napp = FastAPI(docs_url=None, redoc_url=None, openapi_url=None, lifespan=lifespan)\n\n\ndef authorize(authorization: str = Header(default="")):\n    if not secrets.compare_digest(\n        authorization, f"Bearer {state.get(\'key\', \'\')}"\n    ) or not state.get("key"):\n        raise HTTPException(401, "Unauthorized")\n\n\nclass Message(BaseModel):\n    role: str\n    content: str = Field(max_length=100000)\n\n\nclass CompletionRequest(BaseModel):\n    model: str\n    messages: list[Message] = Field(min_length=1, max_length=32)\n    max_tokens: int = Field(default=256, ge=1, le=512)\n    temperature: float = Field(default=0, ge=0, le=2)\n    stream: bool = False\n    response_format: dict | None = None\n\n\n@app.get("/health", dependencies=[Depends(authorize)])\ndef health():\n    return {\n        "ready": state["ready"],\n        "model": MODEL,\n        "gpu": state.get("gpu"),\n        "rocm": state.get("rocm"),\n        "active_requests": len(state["active_models"]),\n        "capacity": len(model_ids()),\n        "models": [\n            {\n                "id": model_id,\n                "revision": state.get("models", {}).get(model_id, {}).get("revision"),\n                "active_requests": int(model_id in state["active_models"]),\n                "capacity": 1,\n            }\n            for model_id in model_ids()\n        ],\n    }\n\n\n@app.get("/v1/models", dependencies=[Depends(authorize)])\ndef models():\n    return {\n        "object": "list",\n        "data": [\n            {"id": model_id, "object": "model", "owned_by": "faultline-amd"}\n            for model_id in model_ids()\n        ],\n    }\n\n\ndef generate(request):\n    import torch\n\n    selected = state["models"][request.model]\n    tokenizer = selected["tokenizer"]\n    model = selected["model"]\n    prompt = tokenizer.apply_chat_template(\n        [message.model_dump() for message in request.messages],\n        tokenize=False,\n        add_generation_prompt=True,\n        enable_thinking=False,\n    )\n    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)\n    input_tokens = inputs.input_ids.shape[1]\n    if input_tokens + request.max_tokens > 8192:\n        raise HTTPException(413, "Prompt and output exceed 8192 tokens")\n    with torch.inference_mode():\n        output = model.generate(\n            **inputs,\n            max_new_tokens=request.max_tokens,\n            do_sample=False,\n            pad_token_id=tokenizer.eos_token_id,\n        )\n    content = tokenizer.decode(\n        output[0][input_tokens:], skip_special_tokens=True\n    ).strip()\n    if request.response_format and request.response_format.get("type") == "json_object":\n        try:\n            if not isinstance(json.loads(content), dict):\n                raise ValueError()\n        except ValueError:\n            raise HTTPException(502, "Model did not return a JSON object") from None\n    output_tokens = output.shape[1] - input_tokens\n    return {\n        "id": f"amd-{secrets.token_hex(8)}",\n        "object": "chat.completion",\n        "created": int(time.time()),\n        "model": request.model,\n        "choices": [\n            {\n                "index": 0,\n                "message": {"role": "assistant", "content": content},\n                "finish_reason": "stop"\n                if output_tokens < request.max_tokens\n                else "length",\n            }\n        ],\n        "usage": {\n            "prompt_tokens": input_tokens,\n            "completion_tokens": output_tokens,\n            "total_tokens": input_tokens + output_tokens,\n        },\n    }\n\n\n@app.post("/v1/chat/completions", dependencies=[Depends(authorize)])\nasync def completions(request: CompletionRequest):\n    if request.model not in model_ids() or request.stream:\n        raise HTTPException(400, "Unsupported model or streaming request")\n    if request.model in state["active_models"] or (not DUAL_MODELS and state["active"]):\n        raise HTTPException(429, "AMD server at capacity")\n    state["active"] = True\n    state["active_models"].add(request.model)\n    task = asyncio.create_task(asyncio.to_thread(generate, request))\n    task.add_done_callback(\n        lambda completed: finish_generation(completed, request.model)\n    )\n    return await asyncio.shield(task)\n\n\ndef finish_generation(task, model):\n    state["active_models"].discard(model)\n    state["active"] = bool(state["active_models"])\n    if not task.cancelled():\n        task.exception()\n')
(setup_dir / 'qwen-models.json').write_text('{\n  "institution": {\n    "model": "Qwen/Qwen3-8B",\n    "revision": "b968826d9c46dd6066d109eabc6255188de91218"\n  },\n  "citizen": {\n    "model": "Qwen/Qwen3-4B",\n    "revision": "1cfa9a7208912126459214e8b04321603b3df60c"\n  }\n}\n')
(setup_dir / 'amd-requirements.txt').write_text('transformers==4.57.1\naccelerate==1.10.1\nfastapi==0.119.0\nuvicorn==0.37.0\n')
install_log = (setup_dir / 'install.log').open('w')
setup_process = subprocess.Popen([sys.executable, '-m', 'pip', 'install', '-r', str(setup_dir / 'amd-requirements.txt')], stdout=install_log, stderr=subprocess.STDOUT)
print('Dependency installer started:', setup_process.pid)


In [ ]:
print('Installer exit code:', setup_process.poll())
print((setup_dir / 'install.log').read_text()[-2000:])


## Start both models

This stops the prior Faultline model server before loading the new pair. GPU loading needs at least 28 GiB free memory and can take several minutes.


In [ ]:
import signal, socket
if setup_process.poll() != 0:
    raise RuntimeError('Wait for installer exit code 0 before starting')
old_processes = []
for process_path in Path('/proc').iterdir():
    if not process_path.name.isdigit():
        continue
    try:
        arguments = (process_path / 'cmdline').read_bytes().decode().split(chr(0))
        working_dir = (process_path / 'cwd').resolve()
        if 'amd_model_server:app' in arguments and '8180' in arguments and working_dir == setup_dir:
            old_processes.append(int(process_path.name))
    except (OSError, UnicodeError):
        continue
for pid in old_processes:
    os.kill(pid, signal.SIGTERM)
deadline = time.monotonic() + 60
while time.monotonic() < deadline:
    with socket.socket() as connection:
        connection.settimeout(1)
        if connection.connect_ex(('127.0.0.1', 8180)) != 0:
            break
    time.sleep(1)
else:
    raise RuntimeError('Port 8180 remains occupied; inspect the previous process')
key_path = setup_dir / 'api-key'
if not key_path.exists():
    descriptor = os.open(key_path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(descriptor, 'w') as handle:
        handle.write(secrets.token_urlsafe(48))
key_path.chmod(0o600)
server_environment = {**os.environ, 'AMD_DUAL_MODELS': 'true', 'AMD_MODEL': 'Qwen/Qwen3-8B', 'AMD_MODEL_MANIFEST': str(setup_dir / 'qwen-models.json'), 'AMD_KEY_FILE': str(key_path)}
server_log = (setup_dir / 'server.log').open('w')
server_process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'amd_model_server:app', '--host', '127.0.0.1', '--port', '8180', '--no-access-log'], cwd=setup_dir, env=server_environment, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
print('Both Hugging Face models are loading. PID:', server_process.pid)


In [ ]:
import urllib.request, urllib.error
api_key = key_path.read_text().strip()
def local_request(path, payload=None, key=api_key):
    request = urllib.request.Request('http://127.0.0.1:8180' + path, data=json.dumps(payload).encode() if payload is not None else None, headers={'Authorization': 'Bearer ' + key, 'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.load(response)
try:
    health_result = local_request('/health')
    print(health_result)
except OSError:
    print('Still loading or failed. Process exit:', server_process.poll())
    print((setup_dir / 'server.log').read_text()[-2000:])


In [ ]:
health_result = local_request('/health')
if not health_result['ready']:
    raise RuntimeError('Wait for ready true')
expected = json.loads((setup_dir / 'qwen-models.json').read_text())
reported = {entry['id']: entry['revision'] for entry in health_result['models']}
if reported != {entry['model']: entry['revision'] for entry in expected.values()}:
    raise RuntimeError('Loaded model revisions do not match the shared manifest')
try:
    local_request('/v1/models', key='incorrect-key')
except urllib.error.HTTPError as error:
    if error.code != 401:
        raise
else:
    raise RuntimeError('Authentication check failed')
for group, entry in expected.items():
    body = {'model': entry['model'], 'messages': [{'role': 'user', 'content': 'Choose the only available action, hold. Return exactly a JSON object with choice 0 and a short nonempty reason.'}], 'max_tokens': 128, 'temperature': 0, 'chat_template_kwargs': {'enable_thinking': False}, 'response_format': {'type': 'json_object'}}
    started = time.perf_counter()
    result = local_request('/v1/chat/completions', body)
    action = json.loads(result['choices'][0]['message']['content'])
    if set(action) != {'choice', 'reason'} or type(action['choice']) is not int or action['choice'] != 0 or not isinstance(action['reason'], str) or not 1 <= len(action['reason'].strip()) <= 240:
        raise RuntimeError('Decision validation failed for ' + group)
    print(group, entry['model'], 'valid decision in', round(time.perf_counter() - started, 2), 'seconds')
print('Both pinned models and authentication verified')


## Endpoint and private configuration

Reuse the existing tunnel when upgrading this instance. For a fresh instance, enable the exposure cell. Set AMD_ORIGIN below to the tunnel's HTTPS origin, then download the generated connection file without sharing its contents.


In [ ]:
EXPOSE_ENDPOINT = False
if EXPOSE_ENDPOINT:
    subprocess.run(['/var/run/secrets/frp-self-service/install'], check=True)
    tunnel = str(Path.home() / '.local/bin/rc-tunnel')
    tunnel_environment = {key: value for key, value in os.environ.items() if key.lower() not in ('http_proxy', 'https_proxy', 'all_proxy')}
    result = subprocess.run([tunnel, 'expose', '--port', '8180'], env=tunnel_environment, capture_output=True, text=True, check=True)
    print(result.stdout)
else:
    print('Reuse the existing HTTPS tunnel to port 8180, or enable exposure for a fresh instance')


In [ ]:
from urllib.parse import urlsplit
from IPython.display import FileLink, display
AMD_ORIGIN = ''
address = urlsplit(AMD_ORIGIN)
if address.scheme != 'https' or not address.hostname or not address.hostname.endswith('.radeon.firstdg.ai') or address.username or address.password or address.query or address.fragment or address.path not in ('', '/'):
    raise ValueError('Set AMD_ORIGIN to the HTTPS tunnel origin')
values = {'AMD_ENABLED': 'true', 'AMD_MODEL': 'Qwen/Qwen3-8B', 'AMD_DECISION_GROUP': 'institution', 'AMD_CAPACITY': '1', 'AMD_BASE_URL': AMD_ORIGIN.rstrip('/') + '/v1', 'AMD_API_KEY': api_key, 'AMD_CITIZEN_ENABLED': 'true', 'AMD_CITIZEN_MODEL': 'Qwen/Qwen3-4B', 'AMD_CITIZEN_DECISION_GROUP': 'citizen', 'AMD_CITIZEN_CAPACITY': '1', 'AMD_CITIZEN_BASE_URL': AMD_ORIGIN.rstrip('/') + '/v1', 'AMD_CITIZEN_API_KEY': api_key}
connection_path = Path.cwd() / 'faultline-amd-connection.env'
descriptor = os.open(connection_path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
with os.fdopen(descriptor, 'w') as handle:
    handle.write(''.join(key + '=' + value + '\n' for key, value in values.items()))
connection_path.chmod(0o600)
display(FileLink(connection_path.name))
print('Private connection file created; do not print its contents')
